# 🎬 YouTube Automation Tool

1. Run **Cell 1** — installs everything (~2 min, once)
2. Run **Cell 2** — opens the web UI, copy the `gradio.live` link to your phone

In [ ]:
# ── CELL 1: Install (run once, then restart runtime) ─────────────────────────
print('Installing packages... (~2 minutes)')
import subprocess, sys

# Install all packages
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'edge-tts', 'moviepy==1.0.3', 'requests',
                'imageio==2.31.6', 'imageio-ffmpeg',
                'gradio', 'nest-asyncio', '-q'], check=True)

# Force-reinstall Pillow to a clean consistent version (fixes _util conflict)
subprocess.run([sys.executable, '-m', 'pip', 'install',
                '--force-reinstall', 'Pillow==10.4.0', '-q'], check=True)

# Install system packages
subprocess.run(['apt-get', 'install', '-y', 'ffmpeg', 'fonts-dejavu-core'],
               capture_output=True)

print('✅ Done!')
print()
print('⚠️  IMPORTANT: click  Runtime → Restart session  then run Cell 2.')

In [ ]:
# ── CELL 2: Web interface ─────────────────────────────────────────────────────
import asyncio, shutil, subprocess, requests, nest_asyncio, gradio as gr
import numpy as np
from pathlib import Path
from PIL import Image as _PILImage, ImageDraw, ImageFont
import edge_tts

if not hasattr(_PILImage, "ANTIALIAS"):
    _PILImage.ANTIALIAS = _PILImage.LANCZOS

nest_asyncio.apply()  # fixes asyncio conflict inside Colab

VOICES = {
    "Narrator Male (US) — Christopher":  "en-US-ChristopherNeural",
    "Narrator Female (US) — Jenny":      "en-US-JennyNeural",
    "Male (US) — Guy":                   "en-US-GuyNeural",
    "Female (US) — Aria":                "en-US-AriaNeural",
    "Male (UK) — Ryan":                  "en-GB-RyanNeural",
    "Female (UK) — Sonia":               "en-GB-SoniaNeural",
}

FONT_PATHS = [
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
    "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
    "/usr/share/fonts/truetype/freefont/FreeSansBold.ttf",
]

def _get_font(size):
    for fp in FONT_PATHS:
        try:
            return ImageFont.truetype(fp, size)
        except Exception:
            pass
    return ImageFont.load_default()

# ── TTS ───────────────────────────────────────────────────────────────────────
async def _tts_async(text, voice, path):
    words = []
    comm = edge_tts.Communicate(text, voice)
    with open(path, 'wb') as f:
        async for chunk in comm.stream():
            if chunk['type'] == 'audio':
                f.write(chunk['data'])
            elif chunk['type'] == 'WordBoundary':
                words.append({
                    'word':     chunk['text'],
                    'start':    chunk['offset']   / 10_000_000,
                    'duration': chunk['duration'] / 10_000_000,
                })
    return words

def run_tts(text, voice, path):
    loop = asyncio.get_event_loop()
    words = loop.run_until_complete(_tts_async(text, voice, path))
    if not words:
        probe = subprocess.run(
            ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_format", str(path)],
            capture_output=True, text=True,
        )
        import json
        try:
            duration = float(json.loads(probe.stdout)["format"]["duration"])
        except Exception:
            duration = 60.0
        raw = text.split()
        gap = duration / len(raw) if raw else 0.4
        words = [{"word": w, "start": i * gap, "duration": gap * 0.85} for i, w in enumerate(raw)]
        print(f"  (estimated timing, {len(words)} words)")
    return words

# ── B-roll ────────────────────────────────────────────────────────────────────
STOPWORDS = {
    "that","this","with","from","they","their","what","will","never",
    "people","have","been","when","your","more","just","into","over",
    "also","very","some","make","like","time","only","need","most",
    "every","even","than","those","know","well","such","many","would",
}
BROLL_SYNONYMS = {
    "rich": "wealth luxury", "money": "money cash", "invest": "investment finance",
    "business": "business office", "market": "stock market trading",
    "success": "success achievement", "bank": "banking finance",
    "financial": "finance money", "profit": "profit growth",
    "growth": "growth chart", "learn": "education learning",
    "discipline": "focus discipline",
}

def fetch_broll(topic, key, tmp, count=6):
    all_words = topic.lower().split()
    keywords = [BROLL_SYNONYMS.get(w, w) for w in all_words if len(w) > 4 and w not in STOPWORDS][:4]
    clips = []
    for i, kw in enumerate(keywords):
        try:
            r = requests.get(
                'https://api.pexels.com/videos/search',
                headers={'Authorization': key},
                params={'query': kw, 'per_page': 2, 'orientation': 'landscape'},
                timeout=15,
            )
            if r.status_code != 200:
                continue
            for video in r.json().get('videos', []):
                files = video.get('video_files', [])
                hd = [f for f in files if f.get('quality') == 'hd' and f.get('width', 0) >= 1280]
                link = (hd or files or [None])[0]
                if not link:
                    continue
                raw = tmp / f'broll_raw_{i}_{len(clips)}.mp4'
                ready = tmp / f'broll_{i}_{len(clips)}.mp4'
                resp = requests.get(link['link'], stream=True, timeout=60)
                with open(raw, 'wb') as f:
                    for chunk in resp.iter_content(8192):
                        f.write(chunk)
                # Transcode: resize + cinematic grade
                vf = ("scale=1920:1080:force_original_aspect_ratio=increase,"
                      "crop=1920:1080,"
                      "colorchannelmixer=rr=1.06:bb=0.88,"
                      "eq=brightness=-0.12:saturation=1.1")
                result = subprocess.run(
                    ["ffmpeg", "-i", str(raw), "-vf", vf,
                     "-c:v", "libx264", "-preset", "ultrafast",
                     "-pix_fmt", "yuv420p", "-an", "-y", str(ready)],
                    capture_output=True, timeout=180,
                )
                if result.returncode == 0:
                    clips.append(ready)
                    raw.unlink(missing_ok=True)
                    print(f"    clip {len(clips)}: '{kw}'")
                if len(clips) >= count:
                    return clips
        except Exception as e:
            print(f'B-roll error: {e}')
    return clips

# ── Subtitle rendering ────────────────────────────────────────────────────────
def _make_subtitle_clip(text, duration, start, highlight_last=False):
    from moviepy.editor import ImageClip
    W, H = 1920, 1080
    FONT_SIZE, PAD, MAX_W = 64, 18, W - 120
    font = _get_font(FONT_SIZE)

    dummy = ImageDraw.Draw(_PILImage.new("RGB", (W, 10)))
    word_list = text.split()
    lines, line = [], []
    for word in word_list:
        line.append(word)
        if dummy.textbbox((0, 0), " ".join(line), font=font)[2] > MAX_W and len(line) > 1:
            line.pop()
            lines.append(" ".join(line))
            line = [word]
    if line:
        lines.append(" ".join(line))

    LINE_H = FONT_SIZE + 10
    total_h = len(lines) * LINE_H + PAD * 2
    img = _PILImage.new("RGBA", (W, total_h), (0, 0, 0, 0))
    draw = ImageDraw.Draw(img)
    last_word = word_list[-1] if highlight_last and word_list else None

    y = PAD
    for li, ln in enumerate(lines):
        is_last = (li == len(lines) - 1)
        bb = draw.textbbox((0, 0), ln, font=font)
        x = (W - (bb[2] - bb[0])) // 2
        if highlight_last and is_last and last_word and ln.endswith(last_word):
            ln_words = ln.split()
            if len(ln_words) > 1:
                prefix = " ".join(ln_words[:-1]) + " "
                prefix_w = draw.textbbox((0, 0), prefix, font=font)[2]
                for dx, dy in [(-2,2),(2,2),(-2,-2),(2,-2),(0,3)]:
                    draw.text((x+dx, y+dy), prefix, fill=(0,0,0,210), font=font)
                draw.text((x, y), prefix, fill=(255,255,255,255), font=font)
                xh = x + prefix_w
                for dx, dy in [(-2,2),(2,2),(-2,-2),(2,-2),(0,3)]:
                    draw.text((xh+dx, y+dy), last_word, fill=(0,0,0,210), font=font)
                draw.text((xh, y), last_word, fill=(255,215,0,255), font=font)
            else:
                for dx, dy in [(-2,2),(2,2),(-2,-2),(2,-2),(0,3)]:
                    draw.text((x+dx, y+dy), ln, fill=(0,0,0,210), font=font)
                draw.text((x, y), ln, fill=(255,215,0,255), font=font)
        else:
            for dx, dy in [(-2,2),(2,2),(-2,-2),(2,-2),(0,3)]:
                draw.text((x+dx, y+dy), ln, fill=(0,0,0,210), font=font)
            draw.text((x, y), ln, fill=(255,255,255,255), font=font)
        y += LINE_H

    arr = np.array(img)
    clip = ImageClip(arr[:,:,:3])
    mask = ImageClip(arr[:,:,3] / 255.0, ismask=True)
    return (clip.set_mask(mask)
               .set_duration(duration)
               .set_start(start)
               .set_position(("center", H - total_h - 70)))

def _subtitle_clips(words, total):
    """Kinetic text: each word appears one by one, new word highlighted in gold."""
    clips = []
    CHUNK = 5
    groups, group = [], []
    for w in words:
        group.append(w)
        if len(group) >= CHUNK:
            groups.append(group); group = []
    if group:
        groups.append(group)

    for g in groups:
        group_end = min(g[-1]["start"] + g[-1]["duration"] + 0.3, total)
        for n in range(len(g)):
            ws = g[n]["start"]
            we = g[n+1]["start"] if n+1 < len(g) else group_end
            dur = we - ws
            if dur < 0.05:
                continue
            partial = " ".join(w["word"] for w in g[:n+1])
            try:
                clips.append(_make_subtitle_clip(partial, dur, ws, highlight_last=True))
            except Exception as e:
                print(f"subtitle error: {e}")
    return clips

# ── Video assembly ────────────────────────────────────────────────────────────
def make_video(broll, audio_path, words, out_path):
    from moviepy.editor import (
        VideoFileClip, AudioFileClip, ColorClip,
        concatenate_videoclips, CompositeVideoClip,
    )
    SIZE = (1920, 1080)
    audio = AudioFileClip(str(audio_path))
    total = audio.duration

    bg_clips, cur, idx = [], 0.0, 0
    if broll:
        while cur < total:
            p = broll[idx % len(broll)]
            try:
                c = VideoFileClip(str(p), audio=False)
                rem = total - cur
                if c.duration > rem:
                    c = c.subclip(0, rem)
                bg_clips.append(c.set_start(cur))
                cur += c.duration
            except Exception as e:
                print(f'Clip error: {e}')
            idx += 1
            if idx > 500:
                break

    if not bg_clips:
        bg_clips = [ColorClip(SIZE, color=(10, 10, 25), duration=total)]

    bg = concatenate_videoclips(bg_clips, method='compose').set_audio(audio)
    overlay = ColorClip(SIZE, color=(0,0,0), duration=total).set_opacity(0.20)

    print(f"  Building kinetic subtitles for {len(words)} words...")
    subs = _subtitle_clips(words, total)
    print(f"  {len(subs)} subtitle clips ready")

    final = CompositeVideoClip([bg, overlay] + subs, size=SIZE)
    final.write_videofile(str(out_path), fps=30, codec='libx264',
                          audio_codec='aac', preset='medium',
                          threads=2, logger=None)
    audio.close()

# ── Thumbnail ─────────────────────────────────────────────────────────────────
def make_thumbnail(title, out_path):
    SIZE = (1280, 720)
    img  = _PILImage.new('RGB', SIZE, (15, 15, 35))
    draw = ImageDraw.Draw(img)
    for y in range(SIZE[1] // 2, SIZE[1]):
        draw.line([(0, y), (SIZE[0], y)], fill=(0, 0, 0))
    font = _get_font(72)
    words_list = title.upper().split()
    lines, line = [], []
    for w in words_list:
        line.append(w)
        if draw.textbbox((0, 0), ' '.join(line), font=font)[2] > SIZE[0] - 100 and len(line) > 1:
            line.pop(); lines.append(' '.join(line)); line = [w]
    if line:
        lines.append(' '.join(line))
    y = (SIZE[1] - len(lines) * 82) // 2 + 80
    for text in lines:
        bb = draw.textbbox((0, 0), text, font=font)
        x  = (SIZE[0] - (bb[2] - bb[0])) // 2
        draw.text((x + 3, y + 3), text, fill=(0, 0, 0), font=font)
        draw.text((x, y),         text, fill=(255, 220, 50), font=font)
        y += 82
    img.save(str(out_path), 'JPEG', quality=95)

# ── Pipeline ──────────────────────────────────────────────────────────────────
def generate(topic, script, voice_label, pexels_key, progress=gr.Progress()):
    if not topic.strip():
        raise gr.Error('Please enter a topic')
    if not script.strip():
        raise gr.Error('Please enter a script')

    voice = VOICES[voice_label]
    safe  = ''.join(c if c.isalnum() else '_' for c in topic)[:35].lower()
    tmp   = Path('/content/tmp')
    tmp.mkdir(exist_ok=True)
    out_mp4   = Path(f'/content/{safe}.mp4')
    out_thumb = Path(f'/content/{safe}_thumbnail.jpg')

    progress(0.1, desc='Generating voiceover...')
    words = run_tts(script, voice, tmp / 'audio.mp3')
    print(f'TTS: {len(words)} words')

    broll = []
    key = pexels_key.strip() if pexels_key else ''
    if key:
        progress(0.3, desc='Fetching B-roll...')
        broll = fetch_broll(topic, key, tmp)
        print(f'B-roll: {len(broll)} clips')
    else:
        print('No Pexels key — using dark background')

    progress(0.5, desc='Assembling video (3-8 min)...')
    make_video(broll, tmp / 'audio.mp3', words, out_mp4)

    progress(0.9, desc='Creating thumbnail...')
    make_thumbnail(topic, out_thumb)

    shutil.rmtree(tmp, ignore_errors=True)
    progress(1.0, desc='Done!')
    return str(out_mp4), str(out_thumb)

# ── UI ────────────────────────────────────────────────────────────────────────
with gr.Blocks(title='YouTube Automation', theme=gr.themes.Soft()) as app:
    gr.Markdown('# 🎬 YouTube Automation Tool')
    gr.Markdown('Enter your topic and script, then press **Generate Video**.')

    with gr.Row():
        with gr.Column():
            inp_topic  = gr.Textbox(label='📌 Topic',
                                    placeholder='e.g. Quotes of Successful People')
            inp_script = gr.Textbox(label='📝 Script',
                                    placeholder='Paste your full script here...',
                                    lines=10)
            inp_voice  = gr.Dropdown(label='🎙 Voice',
                                     choices=list(VOICES.keys()),
                                     value=list(VOICES.keys())[0])
            inp_pexels = gr.Textbox(label='🎥 Pexels API Key (optional)',
                                    placeholder='Get free key at pexels.com/api',
                                    type='password')
            btn = gr.Button('🚀 Generate Video', variant='primary', size='lg')

        with gr.Column():
            out_video = gr.Video(label='📹 Result Video')
            out_thumb = gr.Image(label='🖼 Thumbnail')

    btn.click(
        fn=generate,
        inputs=[inp_topic, inp_script, inp_voice, inp_pexels],
        outputs=[out_video, out_thumb],
    )

app.launch(share=True)